# Pilot 2025 Data Review

This notebook is the pilot-scale counterpart of the laboratory MBDoE workflow. The objective of this first stage is not parameter estimation yet. The objective is to verify that the data can be loaded, normalized, aligned in time, and interpreted before the calibration and estimability pipeline is re-used.

The pilot system uses natural must, so the future design space is narrower than the synthetic-must laboratory campaign. In exchange, the dataset has richer observations for the aroma mass balance, especially ethyl acetate and isoamyl acetate.

## Model-Relevant Data Mapping

The extended fermentation model uses the following observation groups:

- Primary fermentation states: viable biomass, glucose, fructose, assimilable nitrogen, ethanol, glycerol, and temperature input.
- Secondary states: pyruvic acid and acetaldehyde. Acetic acid is included if available.
- Aroma states: total-equivalent and condenser-equivalent observations for ethyl acetate, isoamyl acetate, and ethyl octanoate.
- Online gas information: CO2 sensor trajectories for the subset of batches with sensor files.

Aroma `*_total` columns are interpreted as the total equivalent concentration in wine plus condenser. Aroma `*_condensate` columns are interpreted as the equivalent must concentration accumulated in the condenser. Therefore the retained wine pool can be reconstructed as `total - condensate` when both are measured.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

SCRIPT_DIR = Path.cwd()
if SCRIPT_DIR.name != 'pilot_2025':
    SCRIPT_DIR = Path('fermentation_model/pilot_2025').resolve()
sys.path.insert(0, str(SCRIPT_DIR))
import pilot_2025_data_loader as loader

data = loader.load_pilot_calibration_data()
raw_co2 = loader.load_all_co2_sensor_data(calibration_data=data)
co2, co2_decisions = loader.curate_co2_sensor_data(raw_co2)
coverage = loader.observation_coverage(data)
density_fit = loader.fit_density_total_sugar(data)
data.shape, raw_co2.shape, co2.shape

## Fermentation Sheets and Observation Coverage

In [ ]:
coverage

## Unit Checks

The `ETANOL` column is interpreted as g/L. Its final values are close to 95 g/L, which is realistic for wine. Interpreting the same values as % v/v would be physically impossible.

In [ ]:
unit_cols = ['batch', 'time_h', 'E_g_l', 'E_percent_vv', 'S_GF_g_l', 'density', 'X_viable_kg_m3', 'X_dry_weight_g_l']
data[unit_cols].groupby('batch').agg(['count', 'min', 'median', 'max'])

## Density as Total Sugar Proxy

Density is useful because it is measured more frequently than glucose and fructose. The linear relationship below estimates the information loss incurred when using density as a proxy for total sugar.

In [ ]:
print(density_fit)
df = data[['batch', 'density', 'S_GF_g_l']].dropna()
fig, ax = plt.subplots(figsize=(8, 6))
for batch, group in df.groupby('batch'):
    ax.scatter(group['density'], group['S_GF_g_l'], label=batch, s=24)
x = pd.Series([df['density'].min(), df['density'].max()])
ax.plot(x, density_fit.predict(x), color='black', linewidth=2)
ax.set_xlabel('Density')
ax.set_ylabel('Glucose + fructose (g/L)')
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=8)
plt.show()

## Batch Trajectories

These plots are intended to detect unit problems, missing-data patterns, and whether pilot trajectories excite the same parameter directions as the laboratory designs.

In [ ]:
states = [
    ('temperature_c', 'Temperature (C)'), ('S_GF_g_l', 'G+F (g/L)'),
    ('YAN_mg_l', 'YAN (mg/L)'), ('X_viable_kg_m3', 'Viable X (kg/m3)'),
    ('E_g_l', 'Ethanol (g/L)'), ('glycerol_g_l', 'Glycerol (g/L)'),
    ('pyruvic_acid_mg_l', 'Pyruvic acid (mg/L)'), ('acetaldehyde_mg_l', 'Acetaldehyde (mg/L)'),
    ('ethyl_acetate_total', 'Ethyl acetate total'), ('ethyl_acetate_condensate', 'Ethyl acetate condensate'),
    ('isoamyl_acetate_total', 'Isoamyl acetate total'), ('isoamyl_acetate_condensate', 'Isoamyl acetate condensate'),
]
for batch, group in data.groupby('batch'):
    fig, axes = plt.subplots(6, 2, figsize=(14, 17), sharex=True)
    for ax, (col, ylabel) in zip(axes.ravel(), states):
        ax.plot(group['time_h'], group[col], marker='o', linewidth=1.2)
        if col == 'YAN_mg_l':
            pulses = group[group['N_pulse_mg_l'].fillna(0).gt(0)]
            for _, pulse in pulses.iterrows():
                ax.axvline(float(pulse['time_h']), color='tab:red', linestyle='--', alpha=0.35)
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)
    axes[-1, 0].set_xlabel('Time (h)')
    axes[-1, 1].set_xlabel('Time (h)')
    fig.suptitle(f'Pilot fermentation {batch}', y=1.0)
    fig.tight_layout()
    plt.show()

## Online CO2 Sensor Files

The raw sensor files are first aligned against the first timestamp in the corresponding fermentation sheet. For calibration, `25150` and `25151` are excluded. For `25171`, the pre-activation segment is removed and an effective CO2 time is created with the sustained activation point as `t = 0`.

In [ ]:
co2_decisions

In [ ]:
co2_summary = co2.groupby('batch').agg(
    n_rows=('co2_raw', 'size'), t_original_min_h=('time_h_original', 'min'), t_original_max_h=('time_h_original', 'max'),
    t_effective_min_h=('time_h_effective', 'min'), t_effective_max_h=('time_h_effective', 'max'),
    co2_min=('co2_raw', 'min'), co2_median=('co2_raw', 'median'), co2_max=('co2_raw', 'max')
).reset_index()
co2_summary

In [ ]:
for batch, group in co2.groupby('batch'):
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(group['time_h_effective'], group['co2_raw'], linewidth=0.8)
    ax.set_title(f'CO2 sensor curated signal: {batch}')
    ax.set_xlabel('Effective CO2 time (h)')
    ax.set_ylabel('Raw CO2 signal')
    ax.grid(True, alpha=0.25)
    plt.show()

## Immediate Technical Conclusions

1. The pilot workbook is usable for the extended model workflow.
2. Ethanol should be treated as g/L in this dataset.
3. The pilot data are especially valuable for ethyl acetate and isoamyl acetate because these are measured repeatedly in several fermentations.
4. The design problem should be reformulated for natural must: temperature policy and nutrient additions are realistic control levers, while arbitrary initial glucose/fructose composition and sugar pulses are not primary levers.
5. The next notebook should reuse the reduced secondary/aroma model residual structure, add pilot batches as a separate dataset group, and compare estimability against the laboratory natural/synthetic datasets.